In [1]:
from multiprocessing import Pool
import subprocess
import yaml
import os
import sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile
import copy
from datetime import datetime
import pandas as pd
from collections import Counter
import numpy as np
import threading
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from joblib import load
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import cv2

In [2]:
output_dir = source_path + '/outputs/fine_tuning/'
script_name = source_path+"/scripts/fine-tuning.py"
input_preprocessed='icdar_train_df_patches_20250515_164130.csv'
selected_model = 'resnet18' #'DeiT-Tiny'  # Example model, can be changed
#selected_model = 'DeiT-Tiny' #'DeiT-Tiny'  # Example model, can be changed
selected_classifier = 'logreg'
list_of_metrics = ['majority_vote', 'weighted_vote', 'most_probable']
is_progressive = False  # Set to True for progressive training
model_mode = 'truncated'  # 'truncation', 'full', 'truncated'
truncation='remove head'
custom_transform = False  # Set to True for custom transforms
transform_mode = 'resize'  # 'train', 'val', 'test', 'resize'

In [3]:
pretrained=True

# preparing layer names

In [7]:
transform = u_transforms.get_transform(selected_model, use_patches=True, custom=custom_transform, mode=transform_mode)
model = model_utils.get_model(name=selected_model, mode=model_mode, pretrained=pretrained, truncation=truncation)
out=model_utils.test_output(224,transform, model,huggingface=False)
in_features = out.shape[1]
print(f"Model {selected_model} output features: {in_features}")

Model resnet18 output features: 512


In [8]:
model = FeatureExtractorWithLogReg(
            backbone=model,
            in_features=in_features,
            num_classes=2) # Assuming binary classification

In [9]:
#https://chatgpt.com/share/687124c2-47a8-8010-a4b0-5124a0bf5ecb
all_param_names = [name for name, _ in model.named_parameters()]
backbone_param_names = [name for name, _ in model.named_parameters() if name.startswith('backbone.')]
classifier_param_names = [name for name, _ in model.named_parameters() if name.startswith('classifier.')]

In [10]:
print(f"Total parameters: {len(all_param_names)}")
print(f"Backbone parameters: {len(backbone_param_names)}")
print(f"Classifier parameters: {len(classifier_param_names)}")

Total parameters: 62
Backbone parameters: 60
Classifier parameters: 2


In [11]:
print(f"Backbone parameters: {backbone_param_names}")

Backbone parameters: ['backbone.conv1.weight', 'backbone.bn1.weight', 'backbone.bn1.bias', 'backbone.layer1.0.conv1.weight', 'backbone.layer1.0.bn1.weight', 'backbone.layer1.0.bn1.bias', 'backbone.layer1.0.conv2.weight', 'backbone.layer1.0.bn2.weight', 'backbone.layer1.0.bn2.bias', 'backbone.layer1.1.conv1.weight', 'backbone.layer1.1.bn1.weight', 'backbone.layer1.1.bn1.bias', 'backbone.layer1.1.conv2.weight', 'backbone.layer1.1.bn2.weight', 'backbone.layer1.1.bn2.bias', 'backbone.layer2.0.conv1.weight', 'backbone.layer2.0.bn1.weight', 'backbone.layer2.0.bn1.bias', 'backbone.layer2.0.conv2.weight', 'backbone.layer2.0.bn2.weight', 'backbone.layer2.0.bn2.bias', 'backbone.layer2.0.downsample.0.weight', 'backbone.layer2.0.downsample.1.weight', 'backbone.layer2.0.downsample.1.bias', 'backbone.layer2.1.conv1.weight', 'backbone.layer2.1.bn1.weight', 'backbone.layer2.1.bn1.bias', 'backbone.layer2.1.conv2.weight', 'backbone.layer2.1.bn2.weight', 'backbone.layer2.1.bn2.bias', 'backbone.layer3.0.c

# launch fine-tuning

In [ ]:
total_epochs = 110  # Total number of epochs for fine-tuning
weight_decay = 1e-4  # Weight decay for the optimizer 
pretrain_head=5
lr_classific_head= 1e-3  # Learning rate for the classification head
lr_backbone_initial = 1e-5  # Learning rate for the backbone
lr_backbone_final = 1e-8  # Final learning rate for the backbone
#for vit and transformers choose 0.05
if selected_model == 'resnet18':
    steps=['layer4','layer3','layer2','layer1']
elif selected_model == 'DeiT-Tiny':
    steps=[f'layer.{i}'for i in range(11,-1,-1)]

#for progressive fine tuning
optimizer_phases = [pretrain_head]
phase_layers_to_freeze = [backbone_param_names]
phase_lr = [lr_classific_head]
step_phase=8
for i,name in enumerate(steps):
    optimizer_phases.append(optimizer_phases[i]+step_phase) 
    phase_layers=phase_layers_to_freeze[i]
    phase_layers_to_freeze.append([l for l in phase_layers if not(name in l)])
    if i == 0:
        phase_lr.append(lr_backbone_initial)
    else:
        phase_lr.append(phase_lr[i] * 0.1)  # Decrease learning rate for each phase
n = len(steps)+1
#lr_backbone = [lr_backbone_initial + (lr_backbone_final - lr_backbone_initial) * i / (n - 1) for i in range(n)]
phase_lr += [phase_lr[-1] * 0.5] 
phase_optimizer_name='Adam' #'AdamW
optim_config_progressive = {
        'optimizer_phases':optimizer_phases+[total_epochs],  # Example: [10, 10, 80] for 100 epochs
        'phase_layers_to_freeze':phase_layers_to_freeze+[[]],
        'phase_scheduling': ['no_scheduling' for _ in range(len(optimizer_phases)+1)],
        'phase_optimizer':['Adam' for _ in range(len(optimizer_phases)+1)],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
        'phase_lr': phase_lr,
        #'phase_optimizer_hyperparams': [{'weight_decay':weight_decay} for _ in range(len(optimizer_phases)+1)],
        'phase_optimizer_hyperparams': [{} for _ in range(len(optimizer_phases)+1)],
        'phase_scheduler_hyperparams': [{} for _ in range(len(optimizer_phases)+1)],
    }

#for full fine tuning
optimizer_phases = [pretrain_head, total_epochs]  # Example: [1, 4, 95] for 100 epochs
optim_config_cosine = {
    'optimizer_phases':[5,15,total_epochs],  # Example: [10, 10, 80] for 100 epochs
    'phase_layers_to_freeze':[backbone_param_names,[],[]],
    'phase_scheduling': ['no_scheduling','Linear','CosineScheduleCustom'],
    'phase_optimizer':['AdamW','AdamW','AdamW'],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
    'phase_lr': [1e-3,1e-6,1e-6],
    'phase_optimizer_hyperparams': [{'weight_decay':weight_decay},{'weight_decay':weight_decay},{'weight_decay':weight_decay}],
    'phase_scheduler_hyperparams': [{}, {'warmup_epochs': optimizer_phases[1]}, {'T_max': total_epochs}],
}
optim_config_simple = {
    'optimizer_phases':optimizer_phases,  # Example: [10, 10, 80] for 100 epochs
    'phase_layers_to_freeze':[backbone_param_names,[]],
    'phase_scheduling': ['no_scheduling','no_scheduling'],
    'phase_optimizer':['AdamW','AdamW'],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
    'phase_lr': [1e-3,1e-7],
    'phase_optimizer_hyperparams': [{'weight_decay':weight_decay},{'weight_decay':weight_decay}],
    'phase_scheduler_hyperparams': [{}, {}],
}

optimizer_phases = [total_epochs]  # Example: [1, 4, 95] for 100 epochs
optim_config_scratch = {
    'optimizer_phases':optimizer_phases,  # Example: [10, 10, 80] for 100 epochs
    'phase_layers_to_freeze':[[]],
    'phase_scheduling': [],
    'phase_optimizer':['AdamW'],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
    'phase_lr': [1e-3],
    'phase_optimizer_hyperparams': [{'weight_decay':weight_decay},{'weight_decay':weight_decay}],
    'phase_scheduler_hyperparams': [{}, {}],
}

In [13]:
#parameters
args = DotDict(
    N_max=282,
    patches=True,
    input_filename=input_preprocessed,
    huggingface=False,
    pooling=False,  # if true in transformer models use pooling, if false only the cls token
    custom_transform=custom_transform,  # custom transform for the dataset
    transform_mode=transform_mode,  # 'train', 'val', 'test'
    save_h5=False,
    selected_model=selected_model,  # googlenet, alexnet
    truncation=truncation,
    running='new-laptop',
    saved='old-laptop',
    model_mode=model_mode,  # 'truncation
    batch_size=64,
    select_cls=False,
    num_workers=4,
    pin_memory=True,
    show_image=False,
    checkpoint_path = output_dir+"checkpoint.pt",
    save_path = output_dir,
    total_epochs = total_epochs,
    log_grad_norm = True,
    use_profiler = False,
    run_epochs = 110,
    plot_every = 1,
    patience = 50,
    use_amp = False ,#mixed precision training,
    val_percentage= 1.0 ,#percentage of validation data used for linear evaluation,
    n_splits = 4,
    loss_criterion = 'CrossEntropyLoss',
    selected_classifier=selected_classifier,  # 'logreg', 'svm', 'rf', 'gbc', 'mlp', 'dt',
    optim_config = optim_config_progressive
)

In [ ]:
run_experiment_threaded(args,script_name)  # Test a single run first

Starting experiment: file=icdar_train_df_patches_20250515_164130.csv, model=resnet18
[STDOUT] Output shape:  torch.Size([1, 512])
[STDOUT] tensor([[0.0087, 0.5111]])
[STDOUT] Device is:  cuda
[STDOUT] val writers are:  {np.int64(256), np.int64(259), np.int64(4), np.int64(136), np.int64(137), np.int64(138), np.int64(139), np.int64(13), np.int64(141), np.int64(14), np.int64(16), np.int64(17), np.int64(143), np.int64(275), np.int64(274), np.int64(149), np.int64(23), np.int64(280), np.int64(151), np.int64(24), np.int64(29), np.int64(162), np.int64(34), np.int64(36), np.int64(37), np.int64(167), np.int64(41), np.int64(45), np.int64(173), np.int64(175), np.int64(48), np.int64(177), np.int64(50), np.int64(52), np.int64(53), np.int64(182), np.int64(184), np.int64(58), np.int64(186), np.int64(187), np.int64(190), np.int64(64), np.int64(194), np.int64(195), np.int64(72), np.int64(203), np.int64(206), np.int64(80), np.int64(82), np.int64(84), np.int64(215), np.int64(88), np.int64(217), np.int64(9

# functions

## reload

In [4]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod = reload_modules()

## scripts

In [5]:
def run_experiment_threaded(try_args, script_name):
    def stream_output(pipe, name):
        for line in iter(pipe.readline, ''):
            if line:
                print(f"[{name}] {line}", end='')

    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.yaml') as tmp:
        yaml.dump(try_args.__dict__, tmp)
        tmp_path = tmp.name

    print(f"Starting experiment: file={try_args.input_filename}, model={try_args.selected_model}")

    process = subprocess.Popen(
        ['python', script_name, '--config', tmp_path],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    # Start threads for live output
    stdout_thread = threading.Thread(target=stream_output, args=(process.stdout, 'STDOUT'))
    stderr_thread = threading.Thread(target=stream_output, args=(process.stderr, 'STDERR'))
    stdout_thread.start()
    stderr_thread.start()

    process.wait()
    stdout_thread.join()
    stderr_thread.join()

    print(f"Experiment finished with return code: {process.returncode}")
    return
class DotDict:
    def __init__(self, **entries):
        self.__dict__.update(entries)

    def __setitem__(self, key, value):
        setattr(self, key, value)

    def __getitem__(self, key):
        return getattr(self, key)

    def __repr__(self):
        return f"{self.__dict__}"
def load_config(path):
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
        return DotDict(**config)

## others

In [6]:
class FeatureExtractorWithLogReg(nn.Module):
    def __init__(self, backbone, in_features, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)